In [ ]:
from retain_dss.data.schema import MATERIAL_INPUTS, EconomicParams
from retain_dss.models.trainer import load_models
from retain_dss.optimizer.genetic import run_nsga2
from retain_dss.optimizer.route_selector import select_best_route
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
mat_inputs = {k: (v["range"][0] + v["range"][1]) / 2 for k, v in MATERIAL_INPUTS.items()}
mat_inputs.update({"pvc_content": 58, "contamination_level": 4, "moisture_content": 2,
                   "material_age": 7, "tensile_strength_input": 180})
prices = EconomicParams()
print("Material inputs:", mat_inputs)

In [ ]:
models = {r: load_models(r) for r in ["route1", "route2", "route3"]}
results = {}
for route in ["route1", "route2", "route3"]:
    print(f"Optimising {route}...")
    results[route] = run_nsga2(mat_inputs, models[route], route, prices,
                               pop_size=100, n_gen=200, seed=42)
    print(f"  → {len(results[route].X)} Pareto solutions")
selection = select_best_route(results)
print(f"\nRecommended route: {selection['recommended_route']}")
print(f"Hypervolumes: {selection['hypervolumes']}")

In [ ]:
COLORS = {"route1": "#3b82f6", "route2": "#f59e0b", "route3": "#4ade80"}
fig = go.Figure()
for route_name, res in results.items():
    F = res.F
    x = -F[:, 0]   # pvc_purity
    y = -F[:, 2]   # material_quality
    sz = 8 + 20 * (-F[:, 1] - (-F[:, 1]).min()) / ((-F[:, 1]).max() - (-F[:, 1]).min() + 1e-9)
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers",
        marker=dict(size=sz, color=COLORS[route_name], opacity=0.7),
        name=route_name))
    cx, cy = x.mean(), y.mean()
    rx, ry = x.std()*1.5 + 0.5, y.std()*1.5 + 0.5
    t = np.linspace(0, 2*np.pi, 60)
    fig.add_trace(go.Scatter(x=cx+rx*np.cos(t), y=cy+ry*np.sin(t), mode="lines",
        line=dict(color=COLORS[route_name], dash="dash"), showlegend=False))
best_F = selection["best_objectives_raw"]
fig.add_trace(go.Scatter(x=[-best_F[0]], y=[-best_F[2]], mode="markers+text",
    marker=dict(symbol="star", size=20, color="#fcd34d"),
    text=["★ Best"], textposition="top center", name="Optimal"))
fig.update_layout(xaxis_title="PVC Purity (%)", yaxis_title="Material Quality",
                  template="plotly_dark", height=500)
fig.show()